# Amazon ML Challenge 2026: Stage 1 - Preprocessing and Validation Setup

This notebook covers:
1. **Environment Verification** (SageMaker compute, CPU/RAM, libraries)
2. **Data Exploration & Noise Inspection** (Train & Test sets)
3. **Text & Address Normalization** using `src.preprocessing`
4. **Stratified Validation Split Generation** (`scripts/create_val_split.py`)
5. **Format Validation Checks**

In [ ]:
# 1. Environment Verification
import os
import sys
import pandas as pd
import numpy as np

print(f"Python Version: {sys.version}")
print(f"Pandas Version: {pd.__version__}")

# Ensure project root is in sys.path
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(f"Working Directory: {os.getcwd()}")
print(f"Root Directory: {ROOT_DIR}")

In [ ]:
# 2. Inspect Training and Test Data Samples
DATA_DIR = "../student_resource/dataset/train"

s1_sample = pd.read_csv(f"{DATA_DIR}/train_source1.tsv", sep="\t", nrows=5)
gt_sample = pd.read_csv(f"{DATA_DIR}/train_ground_truth.tsv", sep="\t", nrows=5)

print("Source 1 Sample:")
display(s1_sample)

print("Ground Truth Sample:")
display(gt_sample)

In [ ]:
# 3. Test Preprocessing Pipeline on Diverse Records
from src.preprocessing import preprocess_record, clean_business_name, clean_address

samples = [
    {
        "entity_id": "S1-FR-01",
        "business_name": "Société Générale S.A.R.L.",
        "business_address": "12 Bd Haussmann, 75009 Paris",
        "country": "France",
    },
    {
        "entity_id": "S1-IN-01",
        "business_name": "Tata Consultancy Services Pvt. Ltd.",
        "business_address": "TCS House, Raveline St, Fort, Mumbai, Maharashtra 400001",
        "country": "India",
    },
    {
        "entity_id": "S1-US-01",
        "business_name": "Amazon.com Services LLC",
        "business_address": "410 Terry Ave N, Seattle, WA 98109",
        "country": "US",
    },
]

for s in samples:
    cleaned = preprocess_record(s)
    print(f"Original : {s['business_name']} | {s['business_address']}")
    print(f"Cleaned  : {cleaned['clean_name']} [Core: {cleaned['core_name']}] (Suffix: {cleaned['legal_suffix']})")
    print(f"Address  : {cleaned['clean_address']}")
    print(f"Postal   : {cleaned['postal_code']} | Numerics: {cleaned['numeric_tokens']}")
    print("-" * 80)

In [ ]:
# 4. Generate Local Validation Split (e.g. 10,000 S1 entities)
# Run the script to extract a balanced, fast validation dataset
!python ../scripts/create_val_split.py --data-dir ../student_resource/dataset/train --out-dir ../dataset/val_split --num-s1 10000 --num-distractors 30000

In [ ]:
# 5. Verify the Generated Validation Dataset
val_s1 = pd.read_csv("../dataset/val_split/val_source1.tsv", sep="\t")
val_gt = pd.read_csv("../dataset/val_split/val_ground_truth.tsv", sep="\t")

print(f"Validation Source 1 rows: {len(val_s1):,}")
print(f"Validation Ground Truth rows: {len(val_gt):,}")
print(f"Singletons: {(val_gt['matched_entity_ids'].isna() | (val_gt['matched_entity_ids'] == '')).sum():,}")
display(val_s1.head(3))
display(val_gt.head(3))